# LP3 Machine Learning Practical 6 - K-Means Clustering and Elbow Method

**Problem statement:** Implement K-Means clustering / hierarchical clustering on sales_data_sample.csv dataset. Determine the number of clusters using the elbow method. Dataset: https://www.kaggle.com/datasets/kyanyoga/sample-sales-data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

In [ ]:
# Load dataset
df = pd.read_csv("/content/sales_data_sample.csv", encoding='unicode_escape')
df.head()

In [ ]:
# Basic info
print(df.info())
print(df.describe())
print(df.columns)

# Drop non-numeric and irrelevant columns
df_clean = df.select_dtypes(include=[np.number]).dropna()

# Alternatively, choose a few important numerical features
features = df_clean[['QUANTITYORDERED', 'PRICEEACH', 'SALES', 'ORDERNUMBER']]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print("Data shape after scaling:", X_scaled.shape)

In [ ]:
inertia = []
K = range(1, 11)

for k in K:
    model = KMeans(n_clusters=k, random_state=42)
    model.fit(X_scaled)
    inertia.append(model.inertia_)

# Plot Elbow Curve
plt.figure(figsize=(8,5))
plt.plot(K, inertia, marker='o', color='blue')
plt.title("Elbow Method - Optimal K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia (Within-Cluster Sum of Squares)")
plt.grid(True)
plt.show()

In [ ]:
# Choose K based on elbow result (example: K=4)
kmeans = KMeans(n_clusters=4, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Cluster centers
print("Cluster Centers:\n", kmeans.cluster_centers_)

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(x='PRICEEACH', y='SALES', hue='Cluster', data=df, palette='viridis')
plt.title('K-Means Clustering Visualization')
plt.xlabel('Price Each')
plt.ylabel('Sales')
plt.legend(title='Cluster')
plt.show()

In [ ]:
cluster_summary = df.groupby('Cluster')[['SALES', 'PRICEEACH', 'QUANTITYORDERED']].mean()
print("Cluster Summary:\n", cluster_summary)

## Theory and Explanation
The last cell contains the theory, steps, result interpretation and viva questions as comments.

In [ ]:
# =====================================================================
# THEORY AND EXPLANATION - Practical 6: K-Means Clustering and Elbow Method
# =====================================================================
#
# 1. PROBLEM
# Group the sales records into clusters of similar orders using K-Means, and decide
# the number of clusters with the elbow method. This is UNSUPERVISED learning: there
# is no target label, the algorithm finds structure by itself.
#
# 2. KEY THEORY
# a) K-Means clustering
#    Partitions n points into K clusters. Algorithm:
#      1. Choose K and place K initial centroids (sklearn uses k-means++ so the
#         starting centroids are well spread out).
#      2. Assignment step: assign each point to its nearest centroid (Euclidean
#         distance).
#      3. Update step: move each centroid to the mean of the points assigned to it.
#      4. Repeat steps 2 and 3 until the assignments stop changing.
#    It minimises the inertia (within-cluster sum of squares):
#        WCSS = sum over clusters, sum over points in the cluster, of
#               distance(point, centroid)^2
#
# b) Elbow method
#    Run K-Means for K = 1, 2, 3, ... and plot inertia against K. Inertia always
#    falls as K grows (K = n gives 0), but the drop becomes small after the right K.
#    The "elbow", where the curve bends and the gain flattens, is the best K.
#
# c) Why standardise
#    K-Means uses distance. SALES is in thousands while PRICEEACH is under 100, so
#    without scaling SALES would decide almost everything. StandardScaler makes all
#    features comparable.
#
# d) Hierarchical clustering (the alternative named in the question)
#    Agglomerative: start with every point as its own cluster and repeatedly merge
#    the two closest clusters, producing a tree called a dendrogram. Cutting the
#    dendrogram at a chosen height gives the clusters. linkage, dendrogram and
#    fcluster are imported for this, but only the K-Means option is implemented
#    in this notebook.
#
# 3. STEPS (code flow)
#    1. Load sales_data_sample.csv (encoding unicode_escape because of special
#       characters) with 2823 rows and 25 columns.
#    2. Keep numeric columns and pick four features: QUANTITYORDERED, PRICEEACH,
#       SALES, ORDERNUMBER. Standardise them.
#    3. Run K-Means for K = 1 to 10, store the inertia and plot the elbow curve.
#    4. Choose K = 4 from the elbow, fit K-Means and store the label in a new
#       Cluster column.
#    5. Print the cluster centres (in standardised units), plot clusters on a
#       PRICEEACH versus SALES scatter plot and print the average of each cluster.
#
# 4. RESULTS AND HOW TO READ THEM
#    Elbow curve: inertia starts near 11,300 at K = 1 (this equals 2823 points x
#    4 features, as expected for standardised data) and falls steeply until about
#    K = 3 to 4, then declines slowly. The bend is gradual, so K = 3 or 4 are both
#    defensible; K = 4 was chosen.
#
#    Cluster summary (averages):
#    Cluster   SALES      PRICEEACH   QUANTITYORDERED
#    0         3056.84    93.00       28.18
#    1         3304.36    91.95       30.84
#    2         5894.28    97.26       44.95
#    3         1941.14    54.47       35.50
#
#    Cluster 3: low-priced products (average price about 54), low sales value.
#    Cluster 2: high-value orders, large quantities (about 45) at high prices.
#    Clusters 0 and 1: similar mid-sales orders at high unit prices (about 92 to 93).
#    In the scaled cluster centres they differ mostly in the ORDERNUMBER column
#    (0.83 versus -1.01). Order numbers grow with time, so these two clusters
#    separate mainly by when the order was placed. ORDERNUMBER is really an
#    identifier, so removing it would give more meaningful business clusters.
#    In the scatter plot cluster 3 (yellow) sits at low price, cluster 2 (green)
#    holds the very high sales points, and clusters 0 and 1 overlap because they
#    differ on a feature that is not plotted.
#
# 5. VIVA QUESTIONS
#    Q: Supervised versus unsupervised learning?
#    A: Supervised uses labelled data (target known). Unsupervised has no labels
#       and finds patterns such as clusters.
#    Q: What is inertia?
#    A: The sum of squared distances of points to their cluster centroid. Lower
#       means tighter clusters.
#    Q: How does the elbow method work?
#    A: Plot inertia against K and choose the K where the curve bends.
#    Q: Limitations of K-Means?
#    A: K must be chosen in advance, it is sensitive to outliers and scale, and it
#       assumes roughly round clusters of similar size.
#    Q: K-Means versus hierarchical clustering?
#    A: K-Means needs K upfront and is fast for large data. Hierarchical builds a
#       dendrogram and does not need K upfront but is slower.
#    Q: What is k-means++?
#    A: A smart initialisation that spreads the starting centroids apart, giving
#       better and more stable results.
#    Q: Another way to choose K?
#    A: Silhouette score.